# 🗺️ Búsqueda del Tesoro Vectorial
## Actividad Práctica — Datos Masivos 2026

---

En esta actividad explorarás los operadores fundamentales de **Qdrant** —un motor de búsqueda vectorial—
resolviendo 6 pasos de una búsqueda del tesoro. Cada paso enseña una técnica distinta.

La colección `treasure_hunt_2026` contiene cientos de nodos. La mayoría son **ruido o relleno**;
solo unos pocos son los tesoros reales. Tu trabajo es usar las herramientas correctas para encontrarlos.

### Reglas
- Ejecuta las celdas **en orden** de arriba hacia abajo.
- El **payload** de cada nodo tesoro contiene la pista para el siguiente paso.
- Las celdas con `...` son las que debes completar. Las celdas de **Validación ✅** te confirman si acertaste.
- Guarda los IDs de los nodos encontrados en variables — los necesitarás en el paso 6.

### Hoja de ruta

| Paso | Operador Qdrant | Concepto |
|:----:|----------------|----------|
| 1 | `query_points(using='dense')` | Búsqueda semántica densa |
| 2 | ídem | Semántica: el modelo entiende el significado |
| 3 | `query_points(using='sparse')` | Vectores sparse (SPLADE / keywords) |
| 4 | `query_filter=Filter(must=[...])` | Filtros de metadatos |
| 5 | `limit=K` | Top-K retrieval — simulación de RAG |
| 6 | `RecommendQuery` | Recomendación por ejemplos pos/neg |
| 🐙 | `prefetch` + `FusionQuery(RRF)` | **Bonus** — Búsqueda híbrida |


---
## ⚙️ Setup — Ejecuta primero estas celdas


In [ ]:
# Instalación
!pip install qdrant-client sentence-transformers fastembed

In [ ]:
from qdrant_client import QdrantClient, models
from qdrant_client.models import Filter, FieldCondition, MatchValue, SparseVector
from sentence_transformers import SentenceTransformer
from fastembed import SparseTextEmbedding
import numpy as np

# ── Credenciales ──────────────────────────────────────────────────────────────
# FIXME: reemplaza con las s entregadas por el profesor
QDRANT_URL = "..."
QDRANT_API_KEY = "..."
COLLECTION = "..."
# ─────────────────────────────────────────────────────────────────────────────

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
info = client.get_collection(COLLECTION)
print(f"✅ Conectado. Puntos en colección: {info.points_count}")

In [ ]:
class Embedder:
    """
    Wrapper de vectorización. Usa los MISMOS modelos con los que se pobló la colección.
    No debes modificar esta clase.

    Métodos:
      embedder.dense(text)  -> list[float]  (384 dims, normalizado)
      embedder.sparse(text) -> SparseVector (índices + valores BM25)
    """

    def __init__(self):
        print("Cargando modelo denso  (paraphrase-multilingual-MiniLM-L12-v2) …")
        self._dense = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
        print("Cargando modelo sparse (Qdrant/bm25) …")
        self._sparse = SparseTextEmbedding("Qdrant/bm25")
        print("✅ Modelos listos.")

    def dense(self, text: str) -> list:
        """Embedding denso normalizado de 384 dimensiones (multilingüe)."""
        return self._dense.encode(text, normalize_embeddings=True).tolist()

    def sparse(self, text: str) -> SparseVector:
        """Embedding sparse BM25: devuelve SparseVector(indices, values)."""
        result = next(iter(self._sparse.embed([text])))
        return SparseVector(
            indices=result.indices.tolist(), values=result.values.tolist()
        )


embedder = Embedder()

---
## 🔍 Paso 1 — Búsqueda Densa (Warmup)

### Concepto: embeddings densos

Un **embedding denso** transforma un texto en un vector de números reales (aquí, 384 dimensiones).
El modelo aprende a colocar textos con **significado similar** cerca en el espacio vectorial.

Qdrant usa `client.query_points()` como interfaz unificada de búsqueda:

```python
results = client.query_points(
    collection_name = 'mi_coleccion',
    query          = vector_de_consulta,   # lista de floats
    using          = 'dense',              # qué espacio vectorial usar
    limit          = 5,                    # cuántos resultados devolver
)
# results.points es una lista de ScoredPoint con .id, .score, .payload
```

### Tu tarea

Vectoriza el texto `'Las ideas verdes incoloras duermen furiosamente'` con `embedder.dense()`
y búscalo en la colección. Completa los `???`.


In [ ]:
vec_1 = embedder.dense("...")

results_1 = client.query_points(
    collection_name=COLLECTION,
    query=...,
    using="dense",
    limit=1,
)

# Guarda el ID del nodo encontrado (lo necesitarás en el paso 6)
node_id_step1 = results_1.points[0].id
print(f"Nodo encontrado — ID: {node_id_step1} | Score: {results_1.points[0].score:.4f}")
print(f"Payload: {results_1.points[0].payload}")

In [ ]:
# ✅ Validación Paso 1 ─────────────────────────────────────────────────────────
assert results_1.points, "No obtuviste resultados. ¿Conectaste correctamente?"
assert (
    str(node_id_step1) == "1"
), f'ID incorrecto ({node_id_step1}). Pista: ¿usaste using="dense" y el vector correcto?'
print("✅ Paso 1 completado.")
print(f'\n📌 Pista: {results_1.points[0].payload.get("clue")}')

---
## 📚 Paso 2 — Búsqueda Densa Real

### Concepto: la búsqueda semántica entiende el significado

A diferencia de una búsqueda por palabras clave, la búsqueda densa puede encontrar un documento
aunque uses palabras distintas, siempre que el **significado sea similar**.

Por ejemplo, si buscas `'automobile'` puedes encontrar nodos que dicen `'car'` o `'vehicle'`
porque el modelo sabe que son sinónimos en el espacio vectorial.

In [ ]:
vec_2 = embedder.dense(...)

results_2 = client.query_points(
    collection_name=COLLECTION,
    query=...,
    using=...,
    limit=...,
)

node_id_step2 = results_2.points[0].id
print(f"Nodo encontrado — ID: {node_id_step2} | Score: {results_2.points[0].score:.4f}")
print(f"Payload: {results_2.points[0].payload}")

In [ ]:
# ✅ Validación Paso 2 ─────────────────────────────────────────────────────────
assert results_2.points, "No obtuviste resultados."
assert str(node_id_step2) == "2", (
    f"ID incorrecto ({node_id_step2}). "
    "Asegúrate de usar el texto exacto de la diapositiva de definición."
)
print("✅ Paso 2 completado.")
print(f"\n📌 Pista:")
print(results_2.points[0].payload.get("clue"))

---
## 🔑 Paso 3 — Búsqueda Sparse (BM25)

### Concepto: vectores sparse vs. densos

| | Dense | Sparse |
|--|--|--|
| **Representación** | 384 floats, todos ≠ 0 | Miles de dims, casi todos = 0 |
| **Captura** | Significado semántico | Palabras clave exactas |
| **Útil para** | Sinónimos, paráfrasis | Nombres propios, términos técnicos |

**BM25** (Best Match 25) es el algoritmo de ranking de texto más usado en motores de búsqueda.
Pondera cada término por su frecuencia en el documento (TF) y su rareza en la colección (IDF),
lo que lo hace ideal para encontrar documentos que comparten **palabras clave exactas**.

Para usar búsqueda sparse en Qdrant:
```python
sparse_vec = embedder.sparse(texto)    # devuelve SparseVector(indices, values)
results = client.query_points(
    collection_name = COLLECTION,
    query           = sparse_vec,       # ¡SparseVector, no lista de floats!
    using           = 'sparse',         # espacio sparse
    limit           = 1,
)
```

In [ ]:
query_text_3 = (
    ...
)
sparse_vec_3 = ...
print(f"Sparse vector: {len(sparse_vec_3.indices)} tokens activos (de miles posibles)")

results_3 = client.query_points(
    collection_name=COLLECTION,
    query=...,
    using=...,
    limit=...,
)

node_id_step3 = results_3.points[0].id
print(f"Nodo encontrado — ID: {node_id_step3} | Score: {results_3.points[0].score:.4f}")
print(f"Payload: {results_3.points[0].payload}")

In [ ]:
# ✅ Validación Paso 3 ─────────────────────────────────────────────────────────
assert results_3.points, "No obtuviste resultados."
assert str(node_id_step3) == "3", (
    f"ID incorrecto ({node_id_step3}). "
    'Pista: ¿usaste embedder.sparse() y using="sparse"?'
)
print("✅ Paso 3 completado.")
print(f'\n📌 Pista: {results_3.points[0].payload.get("clue")}')

---
## 🔎 Paso 4 — Búsqueda Densa con Filtros de Metadatos

### Concepto: combinar vectores con filtros

Qdrant permite combinar búsqueda vectorial con **filtros exactos** sobre el payload.
Esto es fundamental: puedes buscar el texto más similar *dentro de un subconjunto* de la colección.

```python
from qdrant_client.models import Filter, FieldCondition, MatchValue

mi_filtro = Filter(
    must=[                                          # AND lógico
        FieldCondition(
            key   = 'campo_del_payload',            # campo a filtrar
            match = MatchValue(value='valor_exacto')
        )
    ]
    # También existen: should (OR) y must_not (NOT)
)

results = client.query_points(
    collection_name = COLLECTION,
    query           = mi_vector,
    using           = 'dense',
    query_filter    = mi_filtro,   # <── aquí aplicas el filtro
    limit           = 1,
)
```

In [ ]:
# Reemplaza los {FIXME_i} con los términos correctos antes de continuar
vec_4 = ...

filtro_4 = Filter(
    must=[
        FieldCondition(
            key=...,
            match=MatchValue(value=...),
        )
    ]
)

results_4 = client.query_points(
    collection_name=COLLECTION,
    query=...,
    using=...,
    query_filter=...,
    limit=1,
)

node_id_step4 = results_4.points[0].id
print(f"Nodo encontrado — ID: {node_id_step4} | Score: {results_4.points[0].score:.4f}")
print(f"Payload: {results_4.points[0].payload}")

In [ ]:
# ✅ Validación Paso 4 ─────────────────────────────────────────────────────────
assert results_4.points, "No obtuviste resultados. ¿El filtro es correcto?"
assert str(node_id_step4) == "4", (
    f"ID incorrecto ({node_id_step4}). "
    'Pista: ¿filtraste por key="class", value="juan"?'
)
print("✅ Paso 4 completado.")
print(f'\n📌 Pista: {results_4.points[0].payload.get("clue")}')

---
## 📄📄📄 Paso 5 — Top-K y Simulación de RAG

### Concepto: recuperar múltiples documentos

En sistemas de **RAG** (Retrieval-Augmented Generation), no se recupera 1 documento sino **K documentos**
que se pasan como contexto a un LLM para que genere una respuesta fundamentada.

El parámetro `limit=K` controla cuántos resultados devuelve Qdrant.

> Puede que nuestras consultas traigan documentos que sean _ruido_ o no tengan que ver con lo que estamos consultando. ¿Y si pruebas con un K más grande para que nuestro _RAG_ mitigue estos _falsos positivos_?

In [ ]:
query_text_5 = ...
vec_5 = ...
filtro_5 = ...

results_5 = client.query_points(
    collection_name=COLLECTION,
    query=vec_5,
    using="dense",
    query_filter=filtro_5,
    limit=5,
)

print(f"Nodos recuperados: {len(results_5.points)}")
for p in results_5.points:
    print(
        f'  ID={p.id} | score={p.score:.4f} | texto: {p.payload.get("text", "")[:80]}...'
    )

In [ ]:
# Reconstruye la pista del Paso 6 ordenando los fragmentos
puntos_ordenados = sorted(
    results_5.points, key=lambda p: p.payload.get("part_order", 99)
)

print("📜 Pista del Paso 6 (lee los fragmentos en orden):\n")
for p in puntos_ordenados:
    orden = p.payload.get("part_order", "?")
    parte = p.payload.get("clue_part", "")
    print(f"  [{orden}] {parte}")

# ✅ Validación Paso 5 ──────────────────────────────────────────────────────────
assert len(results_5.points) == 5, (
    f"Esperaba 5 resultados, obtuve {len(results_5.points)}. "
    'Pista: ¿usaste limit=5 y filtraste por class="agustin"?'
)
ids_encontrados = {str(p.id) for p in results_5.points}
ids_esperados = {"5", "6", "7", "8", "9"}
assert (
    ids_encontrados == ids_esperados
), f"IDs incorrectos: {ids_encontrados}. Revisa el filtro y el limit."
print("\n✅ Paso 5 completado.")

---
## 🧲 Paso 6 — API de Recomendación (¡El Tesoro Final!)

### Concepto: recomendación por ejemplos positivos y negativos

La API `RecommendQuery` de Qdrant no necesita un vector de consulta explícito.
En cambio, le dices: *"quiero algo PARECIDO a estos nodos, pero DIFERENTE a estos otros"*.

Internamente, con la estrategia `AVERAGE_VECTOR`, Qdrant calcula:
```
vector_consulta = mean(positivos) - mean(negativos)
```
y busca los nodos más cercanos a ese vector resultante.

```python
results = client.query_points(
    collection_name = COLLECTION,
    query = models.RecommendQuery(
        recommend = models.RecommendInput(
            positive = [id_a, id_b],   # IDs de nodos que quieres imitar
            negative = [id_c, id_d],   # IDs de nodos que quieres evitar
            strategy = models.RecommendStrategy.AVERAGE_VECTOR,
        )
    ),
    using = 'dense',
    limit = 1,
)
```

In [ ]:
vec_3 = ...

results_6 = client.query_points(
    collection_name=COLLECTION,
    query=models.RecommendQuery(
        recommend=models.RecommendInput(
            positive=[...],
            negative=[...],
            strategy=models.RecommendStrategy.AVERAGE_VECTOR,
        )
    ),
    using=...,
    limit=...,
)

tesoro = results_6.points[0]
print(f"Nodo encontrado — ID: {tesoro.id} | Score: {tesoro.score:.4f}")
print(f"Payload: {tesoro.payload}")

In [ ]:
# ✅ Validación Final ──────────────────────────────────────────────────────────
assert results_6.points, "No obtuviste resultados."
assert str(tesoro.id) == "10", (
    f"ID incorrecto ({tesoro.id}). "
    "Pista: ¿usaste los IDs correctos como positivos (pasos 1 y 2) y negativos (pasos 3 y 4)?"
)
assert tesoro.payload.get("type") == "TESORO_FINAL", "¡Ese no es el tesoro final!"

codigo = tesoro.payload.get("codigo_secreto", "NO_ENCONTRADO")

print("=" * 60)
print("🎉  ¡FELICITACIONES! BÚSQUEDA DEL TESORO COMPLETADA")
print("=" * 60)
print(f'\n{tesoro.payload.get("clue")}')
print(f"\n🔑 Código secreto: {codigo}")
print("\nEntrega este código al profesor para completar la actividad.")

In [ ]:
# Extra
easter_egg = tesoro.payload.get("easter_egg", "NO_ENCONTRADO")
easter_egg

---
## 🐙 Bonus — Búsqueda Híbrida (Dense + Sparse)

### Concepto: lo mejor de dos mundos

La **búsqueda híbrida** combina recuperación densa (semántica) y sparse (keywords) para
obtener mejores resultados que cualquiera de las dos por separado.

Qdrant la implementa con **prefetch** + **RRF** (Reciprocal Rank Fusion):

```python
results = client.query_points(
    collection_name = COLLECTION,
    prefetch = [
        # Paso 1: recupera candidatos con búsqueda densa
        models.Prefetch(query=vec_denso,  using='dense',  limit=10),
        # Paso 2: recupera candidatos con búsqueda sparse
        models.Prefetch(query=vec_sparse, using='sparse', limit=10),
    ],
    # Paso 3: fusiona las dos listas con RRF
    query = models.FusionQuery(fusion=models.Fusion.RRF),
    limit = 1,
)
```

In [ ]:
# search qdrant point with id=73
result_extra = client.retrieve(collection_name=COLLECTION, ids=[73])
result_extra[0].payload["clue"]

In [ ]:
results_ee = ...


nodo_ee = results_ee.points[0]
print(f"Nodo encontrado — ID: {nodo_ee.id}")
print(f"Payload: {nodo_ee.payload}")

In [ ]:
# ✅ Validación Easter Egg ─────────────────────────────────────────────────────
assert results_ee.points, "No obtuviste resultados."
assert (
    str(nodo_ee.id) == "11"
), f"ID incorrecto ({nodo_ee.id}). ¿Usaste ambos prefetch y FusionQuery(RRF)?"
url = nodo_ee.payload.get("result_url", "")
print("🐙 ¡Easter Egg encontrado!")
print(f"Tu recompensa: {url}")